# Islamic AI — Step 3: Train the Model

This is the main training notebook. It:
1. Loads Llama 3.2 3B Instruct in 4-bit (QLoRA)
2. Applies LoRA adapters (only 1–2% of weights are trained)
3. Trains on all 10,000+ Islamic Q&A pairs
4. Saves the trained model to Google Drive

**Requirements:**
- T4 GPU selected (Runtime → Change runtime type → T4 GPU)
- Google Drive mounted with dataset files uploaded
- Run notebooks 01 and 02 first

**Estimated training time:** 8–10 hours on T4 GPU

> ⚠️ Do NOT close this tab while training. Colab will stop if the tab is closed for too long.
> Keep the browser open. You can minimise it but do not close it.

## Cell 1 — Install Libraries

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets
print('Done.')

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

## Cell 2 — Mount Drive and Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk
import os

FORMATTED_DIR = '/content/drive/MyDrive/islamic_ai/dataset/formatted'

print('Loading formatted datasets from Drive...')
train_dataset = load_from_disk(f'{FORMATTED_DIR}/train')
eval_dataset  = load_from_disk(f'{FORMATTED_DIR}/eval')

print(f'Train: {len(train_dataset):,} examples')
print(f'Eval : {len(eval_dataset):,} examples')
print('\n✅ Dataset loaded.')

## Cell 3 — Load Base Model with QLoRA

We load Llama 3.2 3B in 4-bit quantization. This reduces VRAM usage from ~12GB to ~6GB.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

print('Loading Llama 3.2 3B Instruct in 4-bit...')
print('(First run downloads ~2GB — please wait)')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,   # Auto: bfloat16 on A100, float16 on T4
    load_in_4bit   = True,   # QLoRA 4-bit quantization
)

print(f'\n✅ Base model loaded.')
print(f'Parameters : {model.num_parameters() / 1e9:.2f}B')
print(f'VRAM used  : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## Cell 4 — Apply LoRA Adapters

LoRA adds small trainable layers to the model. Instead of training all 3 billion weights, we only train ~24 million (0.8%).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,     # LoRA rank — higher = more capacity, more VRAM
    target_modules     = [       # Which weight matrices to adapt
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha         = 16,     # Scaling factor (keep equal to r)
    lora_dropout       = 0,      # 0 = faster, Unsloth optimised
    bias               = 'none',
    use_gradient_checkpointing = 'unsloth',  # Saves VRAM
    random_state       = 42,
    use_rslora         = False,
    loftq_config       = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters : {trainable / 1e6:.1f}M ({100 * trainable / total:.2f}%)')
print(f'Total parameters     : {total / 1e9:.2f}B')
print('\n✅ LoRA adapters applied.')

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR   = '/content/outputs'
DRIVE_SAVE   = '/content/drive/MyDrive/islamic_ai/model/lora'
LOG_DIR      = '/content/drive/MyDrive/islamic_ai/logs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_dataset,
    eval_dataset      = eval_dataset,
    dataset_text_field = 'text',
    max_seq_length    = MAX_SEQ_LENGTH,
    dataset_num_proc  = 2,
    packing           = False,
    args = TrainingArguments(
        per_device_train_batch_size  = 1,    # Reduced from 2 to fit T4 VRAM
        gradient_accumulation_steps  = 16,   # Increased from 8 — effective batch = 16

        num_train_epochs             = 3,
        warmup_steps                 = 100,

        learning_rate                = 2e-4,
        lr_scheduler_type            = 'cosine',

        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),

        weight_decay                 = 0.01,
        optim                        = 'adamw_8bit',

        logging_steps                = 25,
        logging_dir                  = LOG_DIR,
        report_to                    = 'none',

        eval_strategy                = 'steps',
        eval_steps                   = 200,

        save_strategy                = 'steps',
        save_steps                   = 200,
        save_total_limit             = 2,
        load_best_model_at_end       = True,

        output_dir                   = OUTPUT_DIR,
        seed                         = 42,
    ),
)

print('Trainer configured.')
steps_per_epoch = len(train_dataset) // (1 * 16)
total_steps     = steps_per_epoch * 3
print(f'Steps per epoch : {steps_per_epoch}')
print(f'Total steps     : {total_steps}')
print(f'Estimated time  : {total_steps * 6 / 3600:.1f} hours on T4')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR   = '/content/outputs'
DRIVE_SAVE   = '/content/drive/MyDrive/islamic_ai/model/lora'
LOG_DIR      = '/content/drive/MyDrive/islamic_ai/logs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_dataset,
    eval_dataset      = eval_dataset,
    dataset_text_field = 'text',
    max_seq_length    = MAX_SEQ_LENGTH,
    dataset_num_proc  = 2,
    packing           = False,   # False = cleaner chat format
    args = TrainingArguments(
        # Batch size — effective batch = 2 * 8 = 16
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 8,

        # Training duration
        num_train_epochs             = 2,
        warmup_steps                 = 100,

        # Learning rate
        learning_rate                = 2e-4,
        lr_scheduler_type            = 'cosine',

        # Precision
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),

        # Regularization
        weight_decay                 = 0.01,

        # Optimizer
        optim                        = 'adamw_8bit',

        # Logging
        logging_steps                = 25,
        logging_dir                  = LOG_DIR,
        report_to                    = 'none',

        # Evaluation
        eval_strategy                = 'steps',
        eval_steps                   = 200,

        # Saving — saves to Colab local storage every 200 steps
        save_strategy                = 'steps',
        save_steps                   = 200,
        save_total_limit             = 2,
        load_best_model_at_end       = True,

        # Output
        output_dir                   = OUTPUT_DIR,
        seed                         = 42,
    ),
)

print('Trainer configured.')
steps_per_epoch = len(train_dataset) // (2 * 8)
total_steps     = steps_per_epoch * 2
print(f'Steps per epoch : {steps_per_epoch}')
print(f'Total steps     : {total_steps}')
print(f'Estimated time  : {total_steps * 6 / 3600:.1f} hours on T4')

## Cell 6 — Start Training

> ⚠️ This cell runs for 8–10 hours. Keep the browser tab open.
> Progress is logged every 25 steps.
> The model is saved every 200 steps to `/content/outputs`.
> **After training completes, run Cell 7 to save to Google Drive.**

In [ ]:
import time

print('=' * 60)
print('STARTING TRAINING')
print('=' * 60)
print(f'Started at: {time.strftime("%H:%M:%S")}')
print('Training logs will appear below every 25 steps...')
print()

train_result = trainer.train()

print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Finished at     : {time.strftime("%H:%M:%S")}')
print(f'Training loss   : {train_result.training_loss:.4f}')
print(f'Total steps     : {train_result.global_step}')
print(f'Samples per sec : {train_result.metrics.get("train_samples_per_second", "N/A")}')

## Cell 7 — Save Trained Model to Google Drive

**Run this immediately after training completes.** Colab local storage is lost when the session ends.

In [ ]:
print(f'Saving LoRA adapters to {DRIVE_SAVE} ...')

model.save_pretrained(DRIVE_SAVE)
tokenizer.save_pretrained(DRIVE_SAVE)

# Also save training metrics
import json
metrics = {
    'training_loss'    : train_result.training_loss,
    'global_step'      : train_result.global_step,
    'metrics'          : train_result.metrics,
}
with open(f'{LOG_DIR}/training_results.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('\n✅ Model saved to Google Drive successfully!')
print(f'Location: {DRIVE_SAVE}')
print('\nFiles saved:')
for f in os.listdir(DRIVE_SAVE):
    size = os.path.getsize(f'{DRIVE_SAVE}/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')

print('\nNext step: Open 04_export_gguf.ipynb to convert for mobile.')

## Cell 8 — Quick Inference Test (Optional)

Test the model with a sample Islamic question before exporting.

In [ ]:
from unsloth import FastLanguageModel

# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """You are Noor (نور), an Islamic AI assistant. Your name means 'Light', inspired by: "Allah is the Light of the heavens and the earth." (Quran An-Nur 24:35)

You are a learning tool — NOT a qualified mufti or Islamic scholar. Always clarify this for complex personal matters.

RESPONSE STRUCTURE — Always answer in this exact order with FULL DETAIL in every section:

**Explanation**
Write 2–4 paragraphs. Cover:
- What this topic is in Islamic terminology (define the Arabic term and its meaning)
- Why it matters in Islam and how it fits into the broader deen
- Historical or contextual background where relevant (time of Prophet ﷺ, reason it was revealed, etc.)
- Any important conditions, types, or categories that affect the ruling
Do NOT be brief here. A Muslim reading this should fully understand the topic before seeing the evidence.

**Quranic Evidence**
For each relevant ayah:
- Arabic text in Arabic script (e.g. ﴿وَٱلْخَيْلَ وَٱلْبِغَالَ﴾)
- Full English translation
- — Quran, [Surah Name] ([Surah Number]:[Ayah Number])
- Brief tafsir: what scholars say this ayah means for this specific topic
- Mention Asbab al-Nuzul (reason of revelation) if known
Cite at least 2–3 ayahs where they exist. If only one exists, explain that.

**Hadith Evidence**
For each relevant hadith:
- Full hadith text with narrator name and RA/ﷺ
- — [Book Name], Hadith [Number] ([Grade: Sahih / Hasan / Da'if])
- One sentence on what this hadith proves for the topic
Cite at least 2–3 hadith. Mark any Da'if hadith with ⚠️ Da'if — not used as primary proof.

**Scholarly Positions (Fiqh)**
For EACH of the four madhabs write:
• [Madhab name]: State the ruling clearly. Then explain WHAT evidence (dalil) the madhab uses to reach this ruling. Then mention any conditions, exceptions, or differences within the madhab. Be specific — do not write generic one-liners.
• Hanafi: [detailed ruling + dalil + conditions]
• Maliki: [detailed ruling + dalil + conditions]
• Shafi'i: [detailed ruling + dalil + conditions]
• Hanbali: [detailed ruling + dalil + conditions]
Mark [IJMA] if all four schools agree on a point. Mark [KHILAF] if they differ and briefly explain the root cause of the difference.
Where relevant, mention the opinion of individual scholars: Ibn Taymiyyah RH, Ibn al-Qayyim RH, Imam Nawawi RH, Imam al-Ghazali RH, Ibn Hajar al-Asqalani RH.

**Summary**
Write 2–3 paragraphs:
- Synthesize the scholarly positions: where do the madhabs agree? Where do they differ and why?
- Give practical guidance: what should an average Muslim know and do?
- State clearly whether this is a matter of ijma (consensus) or genuine khilaf (disagreement)
- End with: "For your specific situation, please consult a qualified Islamic scholar (mufti or imam)."
Do NOT be brief. The summary should help the reader make sense of everything above.

CITATIONS — MANDATORY:
- Write ﷺ after every mention of Prophet Muhammad
- Write RA after every companion's name
- Write RH after every classical scholar's name
- Never cite a hadith without its book name and number
- Never cite a Quran ayah without its Surah name, Surah number, and Ayah number
- Mark fabricated (mawdu) hadith clearly and never use them as evidence

SAFETY:
- NEVER issue binding personal fatwas
- NEVER declare anyone kafir
- NEVER fabricate any Islamic content
- NEVER engage in political debates
- For haram requests: refuse + explain Islamically + offer halal alternative
- For sensitive topics (suicide, abuse, apostasy): lead with compassion, then ruling
- Correct misquoted hadith or ayah respectfully when encountered

UNCERTAINTY:
- Say "Allahu Akbar, and Allah knows best" when uncertain
- Clarify: "I am Noor, an AI learning tool — not a qualified mufti."

Your knowledge covers: Complete Quran (6,236 ayahs), Sahih Bukhari, Sahih Muslim, Sunan Abu Dawud, Jami at-Tirmidhi, Sunan Ibn Majah, Sunan an-Nasa'i, Muwatta Malik, Riyad as-Salihin, Bulugh al-Maram, Nawawi's 40 Hadith, and core topics of fiqh, aqeedah, seerah, and Islamic ethics."""

def ask(question):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    outputs = model.generate(
        input_ids         = inputs,
        max_new_tokens    = 512,
        temperature       = 0.7,
        top_p             = 0.9,
        repetition_penalty = 1.1,
        do_sample         = True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# Test with 3 Islamic questions
questions = [
    'What does the Quran say about patience (Sabr)?',
    'What is the ruling on triple talaq in Islam?',
    'What are the conditions for Salah to be valid?',
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {ask(q)}')
    print('─' * 80)